# Human Approval for Irreversible Agent Actions

Anthropic's [autonomy research](https://www.anthropic.com/news/measuring-agent-autonomy) — ~1M production tool calls — found that **~0.8% of agent actions are irreversible** (moving money, deleting data, sending external communication), and that these should *"require mandatory human approval before execution."*

This recipe wires that gate to a **real** trust layer using EMILIA's hardened Python guard, [`examples/grok_guard.py`](https://github.com/emiliaprotocol/emilia-protocol/blob/main/examples/grok_guard.py). The model proposes intent; a **named human** approves the exact action on their own device (WebAuthn / Face ID / passkey), out of band, in a browser; and the agent proceeds **only after that human's device signature has been verified offline** — in the agent's own process, against a **pinned** signer key, **bound** to the exact requested action, **single-use**.

**The one lesson:** approval is not another tool the model is told to call. It is an **executor-side precondition** — `proceed=True` is returned only when a pinned key produced a signature over *this exact* action, and never on the strength of a server saying "approved." The runtime owns authority; the model only proposes.

> Honest scope: the receipt proves a *named, pinned key signed this exact action*, > single-use. It does **not** prove the approver was wise, that the action is lawful, > or that what the human saw on their screen faithfully rendered the bytes that were > signed. Claim only what is enforced.

## 1. Setup

Two pieces:

- **`grok_guard.py`** — the guard. The synchronous path is stdlib-only; `AsyncEmiliaGuard` additionally needs `httpx`.
- **`emilia-verify`** — the pure-Python offline verifier. Without it the guard still runs but refuses to claim a cryptographic guarantee it did not check.

```bash
pip install emilia-verify httpx        # offline verifier (+ httpx for the async guard)
export EP_API_KEY=ep_live_...          # your EMILIA API key (never hardcode it)
```

Nothing in this notebook calls the network: every cell runs against the repo's local JS-signed fixture, the same one the cross-language conformance suite uses.

In [ ]:
# Put the repo's pure-Python verifier and the examples/ package on the path so this
# notebook runs from the repo root with no install. (In your app: pip install emilia-verify
# and import from examples.grok_guard directly.)
import os, sys

REPO = os.path.abspath(os.path.join(os.getcwd(), '..')) \
    if os.path.basename(os.getcwd()) == 'cookbook' else os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'packages', 'python-verify'))
sys.path.insert(0, REPO)

from examples.grok_guard import (
    EmiliaGuard,
    AsyncEmiliaGuard,
    dispatch_emilia_tool,
    release_large_payment,
    resume_release_large_payment,
    fetch_well_known_signer_keys,
    load_trusted_signer_keys,
    expected_from_args,
    _verify_evidence_offline,
    InMemoryReplayStore,
    VerifiedReceipt,
    EMILIA_TOOL_SCHEMA,
)
print('grok_guard imported — public guard API ready')

## 2. Pin the signer — the real defense

The guard does **not** trust the `public_key` that the `/evidence` response served alongside the document. A fully compromised server would just sign a forged receipt with its own key and serve its own pubkey. So the trust root must be **server-independent**: the signing key has to be a member of a set you pinned out of band. With **no** pinned set, the guard fails **closed** (`untrusted_signer`) — it never falls back to the inline key.

There are two ways to supply it:

1. **`EP_TRUSTED_SIGNER_KEYS`** (env) or **`trusted_signer_keys=`** (constructor) — the real defense. A comma-separated list of base64url SPKI keys and/or SHA-256 fingerprints, delivered through a channel the operator does not control.
2. **`fetch_well_known_signer_keys(base_url)`** — a convenience bootstrap that reads the operator's advertised keys from `/.well-known/ep-keys.json`.

**Honest caveat:** the `/.well-known` bootstrap is fetched over https from the *same* operator. It resists wire-tampering and receipt substitution, but a server that controls **both** `/.well-known` **and** `/evidence` could pair a forged key with a forged receipt. Only an **out-of-band pinned set** (`EP_TRUSTED_SIGNER_KEYS` from config management, or a pinned SPKI fingerprint) defends against a fully compromised server.

In [ ]:
# (a) The real defense — an out-of-band pinned set. In production these come from
#     config management / a vault, NOT from the server you are verifying against:
#
#       export EP_TRUSTED_SIGNER_KEYS='<base64url-SPKI>,<sha256-fingerprint>'
#
#     load_trusted_signer_keys() reads that env var (and/or an explicit list) and
#     normalizes it. An empty set => the offline check fails closed.
pinned = load_trusted_signer_keys(env_var='EP_TRUSTED_SIGNER_KEYS')
print('pinned from env:', len(pinned), 'entr(y/ies)')

# (b) Convenience bootstrap (network — shown, not run here). Seed the pinned set from
#     the operator's advertised keys, then hand the result to the guard:
#
#       keys  = fetch_well_known_signer_keys('https://www.emiliaprotocol.ai')
#       guard = EmiliaGuard(trusted_signer_keys=keys)
#
#     Convenience, NOT a full trust root (see the caveat above).
print('fetch_well_known_signer_keys is available for bootstrap:', callable(fetch_well_known_signer_keys))

## 3. The flow — mint, open a signoff, return the approval URL

Inside a live agent tool call you must return **promptly** — you cannot block for minutes while a human picks up their phone. So the default pattern is non-blocking:

1. The agent calls the tool `emilia_require_human_signoff` (schema: `EMILIA_TOOL_SCHEMA`).
2. Your dispatcher runs `dispatch_emilia_tool(args, guard=...)`, which **mints** a pre-action Trust Receipt and **opens a signoff**, then returns immediately with `status="approval_required"` and an **opaque** `approval_url`.
3. Your orchestrator notifies the named human (Slack / SMS / email). They open the URL, review the exact action, and approve with Face ID — on *their* device, in a browser.
4. A later turn (or a worker) resolves the signoff and verifies the signature offline.

`proceed` is `False` at dispatch time — a receipt has been minted, not yet signed. The agent must refuse to execute the irreversible action until a later step returns `proceed=True`.

In [ ]:
# The args the agent's tool call carries (the EMILIA_TOOL_SCHEMA shape). action_type is
# one of the schema's enum values; amount/currency/destination/approver bind the action.
tool_args = {
    'action_type': 'large_payment_release',
    'organization_id': 'org-demo',
    'target_resource_id': 'wire/8841',
    'amount': 82000,
    'currency': 'USD',
    'risk_flags': ['new_destination', 'after_hours'],
    'approver_id': 'operator:iman.schrock',
}

# In a real loop you would build the guard once and pass it in:
#   guard = EmiliaGuard()  # reads EP_API_KEY + EMILIA_BASE_URL + EP_TRUSTED_SIGNER_KEYS
#   result = dispatch_emilia_tool(tool_args, guard=guard, notify=send_to_slack)
# DEFAULT (wait=False) is non-blocking and returns:
#   {'proceed': False, 'status': 'approval_required',
#    'approval_url': 'https://www.emiliaprotocol.ai/signoff/<opaque-id>',
#    'receipt_id': 'tr_...', 'signoff_id': 'so_...'}
# The approval_url carries ONLY an opaque signoff id — no approver email, no PII —
# so it is safe to log, forward, and paste. The named human was bound in the POST body.
print('dispatch returns proceed=False + an opaque approval_url; the agent must NOT execute yet.')
print('schema tool name:', EMILIA_TOOL_SCHEMA['function']['name'])

## 4. Resume and verify offline — `proceed=True` is earned, never asserted

When the signoff resolves, the guard **fetches the signed evidence and verifies the Ed25519 signature offline, in this process**, using the repo's own pure-Python verifier. `verified=True` is returned only when *every* independent check passes — any failure fails **closed**:

| Check | Failure status |
|---|---|
| Ed25519 signature over the canonical EP-RECEIPT-v1 payload verifies | `signature_invalid` |
| Signer key is in the **server-independent pinned set** | `untrusted_signer` |
| Signed `receipt_id` / amount / currency / destination / approver match the request | `claim_mismatch` |
| `receipt_id` is single-use (not already redeemed) | `replay` / `already_consumed` |
| Merkle inclusion proof present + valid (when `require_anchor=True`) | `anchor_required` |
| Evidence carries no signed document | `unsigned_evidence` |
| Verifier not importable | `verifier_unavailable` |

The cell below runs the **exact** offline gate (`_verify_evidence_offline`) against the repo's genuine JS-signed fixture — no network — and shows `proceed=True` only after the signature is verified, the signer is pinned, the claim is bound, and the receipt is unspent. This is the same function the sync and async guards both call, so they cannot drift.

In [ ]:
import json

# Load the genuine JS-signed EP-RECEIPT-v1 fixture + its base64url SPKI public key — the
# same artifact the cross-language conformance suite signs. No network, no API key.
FIX = os.path.join(REPO, 'packages', 'python-verify', 'tests', 'fixtures')
with open(os.path.join(FIX, 'receipt.json')) as f:
    document = json.load(f)
with open(os.path.join(FIX, 'pubkey.txt')) as f:
    public_key = f.read().strip()

# Pin THIS key (the out-of-band trust root). In production: EP_TRUSTED_SIGNER_KEYS.
pinned = load_trusted_signer_keys([public_key])

# The request the agent actually made. expected_from_args() binds the signed claim to it:
# receipt_id (primary), amount, currency, destination, approver must all match.
expected = expected_from_args(
    {
        'action_type': 'large_payment_release',
        'amount': 50000,
        'currency': 'USD',
        'target_resource_id': 'acct_9f12',
        'approver_id': 'operator:iman.schrock',
    },
    'ep_demo_8c1f2a',
)

# The evidence packet, shaped the way /evidence serves it.
evidence = {'document': document, 'public_key': public_key, 'signed': True}

result = _verify_evidence_offline(
    evidence,
    expected=expected,
    trusted_signer_keys=pinned,
    require_anchor=True,          # the fixture carries a valid Merkle anchor
    replay_store=InMemoryReplayStore(),
)
assert isinstance(result, VerifiedReceipt)
print('verified :', result.verified)   # True — signature + pinning + binding + single-use + anchor
print('status   :', result.status)     # 'verified'
print('checks   :', result.checks)      # {'version': True, 'signature': True, 'anchor': True}

# proceed=True is EARNED by this offline check, not by any server status string.
proceed = result.verified is True
print('proceed  :', proceed)

### Wiring the resume step in a real agent

In a real deployment you don't hand-build the evidence packet — the guard fetches it. Two equivalent paths, both ending in the same offline check:

```python
# Synchronous worker / batch job (BLOCKS — never inside a live tool call):
result = dispatch_emilia_tool(tool_args, guard=EmiliaGuard(), wait=True, timeout_s=900)
#   result['proceed'] is True ONLY if the device signature verified offline,
#   the signer was pinned, the signed claim matched tool_args, and the receipt was unspent.

# Async, non-blocking tool + separate resume (the headline pattern):
async with AsyncEmiliaGuard() as guard:
    started = await release_large_payment(tool_args, guard=guard)
    # -> {'status': 'approval_required', 'approval_url': ..., 'receipt_id': ...,
    #     'expected': {...}}  # thread `expected` through so binding is enforced
    # ... human approves on-device, out of band ...
    done = await resume_release_large_payment(
        started['receipt_id'], guard=guard,
        expected=started['expected'],
        execute=move_money,   # runs ONLY on offline-verified approval
    )
    #   done['status'] == 'executed' and done['offline_verified'] is True
    #   only after the signature verified locally.
```

The end-to-end guarantee holds only if your real money-movement call refuses to run unless `proceed=True` (ideally re-verifying at the executor), and you inject a **persistent, atomic** `replay_store` (the executor's DB) — the default `InMemoryReplayStore` is per-process only and is not the production single-use guarantee.

## 5. What this proves — and what it does not

**It proves:** a *named, pinned key* produced an Ed25519 signature over the *exact* canonical action this agent requested — accountable, non-repudiable, request-bound, and single-use. A forged `receipt_status: "approved"` injected on the wire is worthless; a genuinely-signed $1 receipt cannot approve an $82k wire; a real receipt for a different action cannot be substituted; a receipt cannot be redeemed twice. With `EP_TRUSTED_SIGNER_KEYS` configured, a fully compromised EMILIA server cannot make the agent proceed.

**It does NOT prove:**

- that the approver was **wise**, or that approving was the right call;
- that the action is **lawful** or compliant;
- that what the human **saw rendered** on their device faithfully matched the bytes that were signed (WYSIWYS rendering is out of scope of this check);
- anything once `proceed=True` returns — your executor must still refuse to act on anything but `proceed=True`, and the single-use guarantee is only as strong as the `replay_store` you inject.

A few **honest residuals** the guard is explicit about: the default in-memory replay store is per-process (inject a DB-backed atomic store in production); the canonicalizer is not yet RFC 8785 / JCS-strict, so it currently fails **closed** (Python may reject some valid JS-signed receipts, never the reverse — a false-negative risk, not a forgery vector).

## 6. We adversarially red-teamed this

Before this went in front of anyone, the offline gate was red-teamed. Six attack vectors are re-run permanently as a regression suite, [`examples/tests/test_grok_guard_redteam.py`](https://github.com/emiliaprotocol/emilia-protocol/blob/main/examples/tests/test_grok_guard_redteam.py):

| # | Attack | Verdict after hardening |
|---|---|---|
| 1 | Tampered action under the enrolled key | `signature_invalid` |
| 2 | Attacker self-signs, serves its own pubkey (unpinned) | `untrusted_signer` |
| 3 | Genuinely-signed **different** receipt (id / amount / destination) | `claim_mismatch` |
| 5a | Anchor stripped / partial when `require_anchor=True` | `anchor_required` |
| 5b | Same receipt presented twice / `consumed` status | `replay` / `already_consumed` |
| 6 | Hostile evidence bodies (str / int / list / junk / verifier raises) | `verified=False`, never raises |

Plus a control: the genuine fixture with a matching request and the fixture key pinned → `verified=True`. The cell below reproduces two of the vectors inline against the same fixture, no network.

```bash
PYTHONPATH=packages/python-verify python3 examples/tests/test_grok_guard_redteam.py
# or:  PYTHONPATH=packages/python-verify pytest examples/tests/test_grok_guard_redteam.py
```

In [ ]:
import copy

# Vector 3 (request binding): a genuinely-signed $50k receipt is presented to authorize
# an $82k wire. Signature + signer are fine, but the signed amount != requested amount.
expected_82k = expected_from_args(
    {'action_type': 'large_payment_release', 'amount': 82000, 'currency': 'USD',
     'target_resource_id': 'acct_9f12', 'approver_id': 'operator:iman.schrock'},
    'ep_demo_8c1f2a',
)
r3 = _verify_evidence_offline(
    {'document': document, 'public_key': public_key},
    expected=expected_82k, trusted_signer_keys=pinned, replay_store=InMemoryReplayStore(),
)
print('vector 3  ->', r3.status, '| verified:', r3.verified)   # claim_mismatch | False
assert r3.verified is False and r3.status == 'claim_mismatch'

# Vector 2b (secure-by-default): with NO pinned set, even the genuine receipt is
# rejected. The guard never falls back to trusting the inline key.
r2 = _verify_evidence_offline(
    {'document': document, 'public_key': public_key},
    expected=expected, trusted_signer_keys=None, replay_store=InMemoryReplayStore(),
)
print('no pin    ->', r2.status, '| verified:', r2.verified)   # untrusted_signer | False
assert r2.verified is False and r2.status == 'untrusted_signer'

print('\nBoth adversarial vectors fail closed. Full suite: 6 vectors + control, all green.')

## Production references

- The guard, single source of truth: [`examples/grok_guard.py`](https://github.com/emiliaprotocol/emilia-protocol/blob/main/examples/grok_guard.py) (`EmiliaGuard` / `AsyncEmiliaGuard`, `dispatch_emilia_tool`, `release_large_payment` / `resume_release_large_payment`).
- Red-team regression suite: [`examples/tests/test_grok_guard_redteam.py`](https://github.com/emiliaprotocol/emilia-protocol/blob/main/examples/tests/test_grok_guard_redteam.py).
- Offline verifier (`pip install emilia-verify`) and the production-grade `@emilia-protocol/verify` `verifyTrustReceipt()`, which fails closed on a missing inclusion proof (stricter than this example's opt-in `require_anchor`).
- Receipt format: [draft-schrock-ep-authorization-receipts](https://datatracker.ietf.org/doc/draft-schrock-ep-authorization-receipts/) (IETF); the EP-Verified Execution conformance class re-verifies at the executor (Internet-Draft §6.3).
- The strictness threshold maps to the *earned-trust* curve in [Anthropic's autonomy research](https://www.anthropic.com/news/measuring-agent-autonomy): gate everything at first, relax per-action as a clean approval history accrues.